# ATWS — Gate 1 measurement

Hypothesis: `research/hypotheses/atws_zscore_reversion.md`

Two runs on the 2018-01-02 → 2022-12-30 training slice, sharing entries and EOD-flat:

- **Run 1 — Dynamic Regime Exits** (`exit_mode='signal'`): anchor-touch limit TP, close-of-bar z ≤ -2.25 stop with next-bar-open fill.
- **Run 2 — Default ATR wrapper** (`exit_mode='atr'`, 1.5× ATR stop, 2.0× ATR TP): cross-hypothesis baseline on the same entries.

Both runs use 1 contract, $5 RT, max 1 concurrent position, RTH only, EOD-flat at 15:45 NY.

Test slice (2023–2024) is **sealed** — not touched here.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from engine.data_loader import load_csv
from engine.features import build_features, build_atws_features
from engine.gate1 import gate1_evaluate, print_gate1

RNG_SEED = 42
BOOTSTRAP_ITERATIONS = 10_000

## Load data and slice to training window

In [ ]:
df_full = load_csv(str(PROJECT_ROOT / 'data' / 'nq_15m_data.csv'), session_filter=True)
df = df_full.loc['2018-01-02':'2022-12-30'].copy()
print(f'Training slice: {df.index.min()}  →  {df.index.max()}')
print(f'Bars: {len(df):,}')

## Build features and signals

Locked parameters from the pre-reg:
- `anchor_window = 20`
- `entry_z = -1.5`
- `stop_distance_z = 0.75` → dynamic stop at z ≤ -2.25
- Entry window: 10:00 ≤ t ≤ 13:00 NY
- Direction: LONG only

In [ ]:
ANCHOR_WINDOW = 20
ENTRY_Z = -1.5
STOP_DISTANCE_Z = 0.75
STOP_Z_THRESHOLD = ENTRY_Z - STOP_DISTANCE_Z  # -2.25
ENTRY_START = pd.Timestamp('10:00').time()
ENTRY_END = pd.Timestamp('13:00').time()

sigs = build_features(df)
sigs = build_atws_features(sigs, anchor_window=ANCHOR_WINDOW)

time_vals = sigs.index.time
in_window = (time_vals >= ENTRY_START) & (time_vals <= ENTRY_END)

entry_mask = (sigs['signal_zscore'] <= ENTRY_Z) & in_window
sigs['signal'] = entry_mask.fillna(False).astype(np.int8)

sigs['exit_tp_price'] = sigs['target_price']
sigs['exit_signal_stop'] = (sigs['z_score'] <= STOP_Z_THRESHOLD).fillna(False)

print(f'Total entry-eligible bars: {int(sigs["signal"].sum()):,}')
print(f'Bars where z≤-2.25 (stop trigger condition fires somewhere): {int(sigs["exit_signal_stop"].sum()):,}')

## Run 1 — Dynamic Regime Exits

In [ ]:
run1_overrides = {
    'exit_mode': 'signal',
    'max_concurrent_trades': 1,
}
g1_run1 = gate1_evaluate(sigs, wrapper_overrides=run1_overrides)
print_gate1(g1_run1)

## Run 2 — Default ATR wrapper (1.5× / 2×)

In [ ]:
run2_overrides = {
    'exit_mode': 'atr',
    'max_concurrent_trades': 1,
}
g1_run2 = gate1_evaluate(sigs, wrapper_overrides=run2_overrides)
print_gate1(g1_run2)

## Supplementary metrics (per pre-reg)

Trade count, win rate, gross profit vs gross loss, t-stat / expectancy with bootstrap 95% CI — for both runs.

In [ ]:
def trade_pnls(result):
    return np.array([t.pnl for t in result.backtest.trades]) if result.backtest.trades else np.array([])

def supplementary(pnls, label):
    n = len(pnls)
    if n == 0:
        print(f'{label}: no trades')
        return
    wins = pnls[pnls > 0]
    losses = pnls[pnls < 0]
    gp = wins.sum()
    gl = losses.sum()  # negative
    win_rate = len(wins) / n
    expectancy = pnls.mean()
    sd = pnls.std(ddof=1) if n > 1 else 0.0
    t_stat = expectancy / (sd / np.sqrt(n)) if sd > 0 else float('nan')

    rng = np.random.default_rng(RNG_SEED)
    boot_means = rng.choice(pnls, size=(BOOTSTRAP_ITERATIONS, n), replace=True).mean(axis=1)
    ci_lo, ci_hi = np.quantile(boot_means, [0.025, 0.975])

    print(f'{label}')
    print(f'  Trade count:           {n}')
    print(f'  Win rate:              {win_rate:.1%}')
    print(f'  Gross profit:          ${gp:,.2f}')
    print(f'  Gross loss:            ${gl:,.2f}')
    print(f'  Net P&L:               ${gp + gl:,.2f}')
    print(f'  Expectancy / trade:    ${expectancy:,.2f}')
    print(f'  95% bootstrap CI:      [${ci_lo:,.2f}, ${ci_hi:,.2f}]   (seed={RNG_SEED}, n_boot={BOOTSTRAP_ITERATIONS:,})')
    print(f'  t-statistic (mean P&L): {t_stat:.3f}')
    print()

In [ ]:
supplementary(trade_pnls(g1_run1), 'Run 1 — Dynamic Regime Exits')
supplementary(trade_pnls(g1_run2), 'Run 2 — Default ATR wrapper')

## Exit-reason breakdown

Sanity check: what fraction of Run 1 trades exited on anchor-touch TP vs regime stop vs EOD?

In [ ]:
def exit_breakdown(result, label):
    trades = result.backtest.trades
    if not trades:
        print(f'{label}: no trades')
        return
    reasons = pd.Series([t.exit_reason for t in trades]).value_counts()
    print(f'{label}')
    print(reasons.to_string())
    print()

exit_breakdown(g1_run1, 'Run 1')
exit_breakdown(g1_run2, 'Run 2')